# Day 3 실습 코드 — CNN · 이미지 분류 · 허깅페이스 데이터

- 강의 자료 [Day 3](https://github.com/Daesung7723/AI-Programming-with-Python/blob/main/Lectures/Day03_CNN_%EC%9D%B4%EB%AF%B8%EC%A7%80%EB%B6%84%EB%A5%98_%ED%97%88%EA%B9%85%ED%8E%98%EC%9D%B4%EC%8A%A4%EB%8D%B0%EC%9D%B4%ED%84%B0.md)의 코드를 강의 순서대로 정리한 노트북입니다.
- 절 제목의 번호는 강의 자료의 절 번호와 같습니다. 개념 설명과 확인 방법은 강의 자료에서 확인합니다.
- 위에서부터 차례로 실행합니다. 뒤의 셀은 앞 셀에서 만든 변수를 사용합니다.
- 응용 실습은 포함하지 않습니다. 강의 자료의 과제를 보고 AI 코딩 도구와 함께 작성합니다.

## 1. 복습과 오늘의 목표

### 1.5 실행 환경 점검 — 수업 시작 전

**② 설치와 확인**

In [ ]:
!pip install -q datasets            # ① 허깅페이스 데이터 라이브러리 설치
import datasets
print(datasets.__version__)         # ② 버전 번호가 출력되면 설치 완료

## 2. 기존 신경망의 한계

### 2.1 평탄화 — 공간 정보의 손실

In [ ]:
import torch
x = torch.arange(16).view(4, 4)     # ① 0~15를 4×4로 — 값 = 원래 위치
print(x)
v = x.flatten()                     # ② 모든 축을 한 축으로
print(v.shape, v)

### 2.2 파라미터 폭증

In [ ]:
import torch, torch.nn as nn

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

fcn1 = nn.Linear(in_features=784, out_features=512)       # 28×28 흑백
fcn2 = nn.Linear(in_features=150528, out_features=512)    # 224×224 컬러
print(f"28x28 흑백:   {count_parameters(fcn1):,}")
print(f"224x224 컬러: {count_parameters(fcn2):,}")

### 2.5 응용 실습 — 채널을 빠뜨린 평탄화

> 오류가 발생하는 것이 정상입니다. 응용 실습 2.5에서 원인을 찾는 대상 코드입니다.

In [ ]:
x = torch.rand(8, 3, 32, 32)          # 32×32 컬러 이미지 8장
fc = nn.Linear(32 * 32, 100)
out = fc(x.flatten(1))                # flatten(1) — 1번 축부터 끝까지 한 축으로(0번 = 이미지 수 유지)

## 3. 합성곱 연산

### 3.1 커널·슬라이딩·특징 지도

**합성곱 실습 2단계 — 코드로 특징 지도 전체 계산**

In [ ]:
import torch.nn.functional as F
img = torch.tensor([[1, 1, 1, 9, 9, 9],
                    [1, 1, 1, 9, 9, 9],
                    [1, 1, 1, 9, 9, 9],
                    [1, 1, 1, 1, 9, 9],
                    [1, 1, 1, 1, 9, 9],
                    [1, 1, 1, 1, 9, 9]], dtype=torch.float32)   # ① 입력 6×6
k = torch.tensor([[-1, 0, 1],
                  [-1, 0, 1],
                  [-1, 0, 1]], dtype=torch.float32)            # ② 커널 — 오른쪽이 밝을수록 큰 값
fm = F.conv2d(img.view(1, 1, 6, 6), k.view(1, 1, 3, 3))       # ③ (배치, 채널, 높이, 너비) 4축으로 바꾸어 합성곱
print(fm.shape)
print(fm[0, 0])                                                # ④ 배치 0번 · 채널 0번의 특징 지도

### 3.6 코드 확인 — 세 가지 합성곱의 출력 크기

In [ ]:
import torch, torch.nn as nn

x = torch.randn(1, 1, 10, 10)                  # (배치, 채널, 높이, 너비)

conv_a = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=0)
conv_b = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1)
conv_c = nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=1)

print("패딩 0 · 스트라이드 1:", conv_a(x).shape)
print("패딩 1 · 스트라이드 1:", conv_b(x).shape)
print("패딩 1 · 스트라이드 2:", conv_c(x).shape)
print("conv_a 가중치:", conv_a.weight.shape)

## 4. 풀링과 계층적 특징

### 4.1 맥스 풀링 — 크기 축소와 핵심 보존

**풀링 실습 1단계 — 3.1 특징 지도에 맥스 풀링 적용**

In [ ]:
pool = nn.MaxPool2d(2)          # ① 2×2 창 · 스트라이드 2
p = pool(fm)                    # ② 3.1 실습의 특징 지도(1, 1, 4, 4)
print(p.shape)
print(p[0, 0])

In [ ]:
for col in [0, 1, 2]:
    m = torch.zeros(1, 1, 4, 4)
    m[0, 0, 1, col] = 9                 # ③ 1행 col열에만 반응
    print(col, pool(m)[0, 0].tolist())

## 5. 데이터 파이프라인

### 5.1 로더가 필요한 이유

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MyImages(Dataset):                   # ① 상속 — Dataset의 규약을 따르는 새 클래스
    def __init__(self, paths, labels, transform=None):
        self.paths, self.labels = paths, labels
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:                 # ② 읽는 순간 변환 적용
            image = self.transform(image)
        return image, self.labels[idx]     # ③ (이미지, 라벨) 튜플

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class Squares(Dataset):
    def __init__(self, n):
        self.n = n                          # ① 샘플 수만 보관 — 데이터를 미리 만들지 않음
    def __len__(self):
        return self.n
    def __getitem__(self, idx):
        x = torch.tensor([float(idx)])      # ② 요청받은 순간 샘플 하나를 만듦
        return x, x ** 2

sq = Squares(10)
print(len(sq))                              # ③ __len__ 호출
print(sq[3])                                # ④ __getitem__(3) 호출

### 5.3 학습용·검증용 변환의 차이와 그 이유

In [ ]:
import numpy as np
from PIL import Image
from torchvision import transforms

arr = np.array([[[255, 0, 0], [0, 255, 0]],
                [[0, 0, 255], [128, 128, 128]]], dtype=np.uint8)   # ① 2×2 RGB — 빨강·초록 / 파랑·회색
img = Image.fromarray(arr)                                         # ② 배열 → PIL 이미지
t = transforms.ToTensor()(img)                                     # ③ 변환 객체를 만들어 이미지에 적용
print(t.shape, t.dtype)
print(t)

**변환 실습 2단계 — `Normalize()`**

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
n = transforms.Normalize(MEAN, STD)(t)                             # ④ 채널별 (값 − 평균) ÷ 표준편차
print(n[0])                                                        # ⑤ 채널 0 평면만 출력

### 5.5 배치와 epoch의 순환

> 반복의 구조만 나타낸 코드이며 따로 실행하지 않습니다(강의 자료 5.5).

```python
for epoch in range(EPOCHS):                # 바깥 반복 = epoch
    for images, labels in train_loader:    # 안쪽 반복 = 배치 — 로더가 (이미지, 라벨) 묶음을 공급
        ...                                # Day 2 6절 학습 루프 5단의 위치
```

**로더 실습 2단계 — 배치로 묶기**

In [ ]:
loader = DataLoader(sq, batch_size=4, shuffle=False)   # ① 1단계의 sq — 샘플 10개
print(len(loader))                                     # ② 배치 수
for xb, yb in loader:
    print(xb.shape, xb.flatten().tolist())             # ③ 배치마다 형상과 입력값

**로더 실습 3단계 — 섞기와 epoch**

In [ ]:
loader = DataLoader(sq, batch_size=4, shuffle=True)    # ④ 매 epoch 순서를 섞음
for epoch in range(2):
    for xb, _ in loader:
        print(epoch, xb.flatten().tolist())            # ⑤ epoch 번호와 배치의 입력값

### 5.6 응용 실습 — 검증 변환을 NumPy로 재현

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
val_tf = transforms.Compose([                 # 5.3 표의 검증 변환 네 단계를 순서대로 묶음
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

### 5.7 응용 실습 — 같은 사진, 다른 결과

> 응용 실습 5.7에서 원인을 찾는 대상 코드입니다. 두 결과가 다르게 출력되는 것이 정상입니다.

In [ ]:
check_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ColorJitter(brightness=0.3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
a, b = check_tf(img), check_tf(img)           # 5.6 ①의 시험 이미지
print(a.shape, b.shape, torch.equal(a, b))

## 6. 실습 ① — 허깅페이스 데이터 준비

### 6.1 허깅페이스 — 공개 모델과 데이터의 저장소

In [ ]:
from datasets import load_dataset

ds = load_dataset("AI-Lab-Makerere/beans")         # ① 세 분할을 한 번에 내려받음 — 약 180MB
print(ds)                                           # ② 분할별 행 수와 열 이름

names = ds["train"].features["labels"].names        # ③ 라벨 번호 → 클래스 이름 목록
print(names)

row = ds["train"][0]                                # ④ 한 행 = 열 이름을 키로 하는 딕셔너리
print(row["labels"], names[row["labels"]], row["image"].size)

### 6.2 클래스별 폴더로 내보내기

**내보내기 실습 1단계 — 클래스별 폴더로 저장**

In [ ]:
import os
ROOT = '/content/data'                                         # ① 내보낼 위치
SPLITS = {'train': 'train', 'validation': 'val', 'test': 'test'}   # ② 허깅페이스 분할 → 폴더 이름

for hf_split, folder in SPLITS.items():
    for i, row in enumerate(ds[hf_split]):
        cls = names[row['labels']]                             # ③ 번호 → 클래스 이름
        dst = os.path.join(ROOT, folder, cls)
        os.makedirs(dst, exist_ok=True)                        # ④ 폴더가 없으면 생성
        row['image'].save(os.path.join(dst, f'{i:04d}.jpg'), quality=95)   # ⑤ 파일로 저장

**참고 코드**

In [ ]:
for folder in ['train', 'val', 'test']:
    for cls in sorted(os.listdir(os.path.join(ROOT, folder))):
        n = len(os.listdir(os.path.join(ROOT, folder, cls)))
        print(f'{folder:5s} {cls:18s} {n:4d}장')

### 6.3 증강 변환 목록과 자르기 범위 판별 활동

In [ ]:
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

cls_dir = os.path.join(ROOT, 'train', 'angular_leaf_spot')
sample = Image.open(os.path.join(cls_dir, sorted(os.listdir(cls_dir))[0]))   # ① 병해 잎 사진 한 장
crop_wide  = transforms.RandomResizedCrop(224, scale=(0.08, 1.0))          # ② 기본값 — 원본 넓이의 8~100%
crop_tight = transforms.RandomResizedCrop(224, scale=(0.7, 1.0))           # ③ 6.4의 값 — 원본 넓이의 70~100%

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax in axes[0]:                                                         # ④ 윗줄 = 기본값
    ax.imshow(crop_wide(sample)); ax.axis('off')
for ax in axes[1]:                                                         # ⑤ 아랫줄 = 0.7
    ax.imshow(crop_tight(sample)); ax.axis('off')
plt.show()

### 6.4 로더 실행과 배치 형상 확인

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    # transforms.RandomHorizontalFlip(),                  # ① 6.3 판정 결과에 따라 주석 해제
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((256, 256)),                        # ② 튜플 — 가로·세로 모두 256
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

**콩잎 로더 1단계 — 데이터셋과 샘플 하나**

In [ ]:
train_ds = datasets.ImageFolder(f'{ROOT}/train', transform=train_tf)
val_ds   = datasets.ImageFolder(f'{ROOT}/val',   transform=val_tf)
print(train_ds.classes, train_ds.class_to_idx)            # ③ 이름 목록 · 이름 → 번호
print(len(train_ds), len(val_ds))                         # ④ 샘플 수

x, y = val_ds[0]                                          # ⑤ 검증 샘플 하나 — 변환을 적용한 결과
print(x.shape, x.dtype, y)
print(x.min().item(), x.max().item())

**콩잎 로더 2단계 — 배치 확인**

In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
print(len(train_loader), len(val_loader))                 # ⑥ 배치 수

images, labels = next(iter(train_loader))                 # ⑦ 첫 배치 하나만 받음
print(images.shape, images.dtype)
print(labels)

### 6.5 허깅페이스에서 데이터를 받을 수 없을 때의 대체 경로

> **교수가 전환을 안내한 경우에만** 실행합니다. CIFAR-10(약 170MB)을 내려받으며, 앞에서 만든 데이터셋이 CIFAR-10으로 교체됩니다(강의 자료 6.5).

In [ ]:
cifar_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                               transforms.Normalize(MEAN, STD)])
train_ds = datasets.CIFAR10(root='/content/cifar', train=True,  download=True, transform=cifar_tf)
val_ds   = datasets.CIFAR10(root='/content/cifar', train=False, download=True, transform=cifar_tf)

## 7. 실습 ② — CNN 정의와 학습

### 7.1 모델 클래스 — `SmallCNN`

In [ ]:
import torch, torch.nn as nn

class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()                                # ① 부모(nn.Module) 초기화 — 반드시 첫 줄
        self.features = nn.Sequential(                    # ② 특징 추출부 — 층을 순서대로 묶음
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(32 * 56 * 56, num_classes)   # ③ 분류부

    def forward(self, x):                                 # ④ 층을 적용하는 순서
        x = self.features(x)
        x = torch.flatten(x, 1)                           # ⑤ 평탄화 — 배치 축(0)은 남기고 나머지를 펴기
        return self.classifier(x)

model = SmallCNN(num_classes=len(train_ds.classes))
print(model)

In [ ]:
x = torch.zeros(1, 3, 224, 224)                       # ① 이미지 1장 크기의 입력(값은 모두 0)
for layer in model.features:                          # ② 특징 추출부의 층을 하나씩 통과
    x = layer(x)
    print(type(layer).__name__, tuple(x.shape))       # ③ 층 이름과 출력 형상
print('flatten', tuple(torch.flatten(x, 1).shape))

**모델 실습 2단계 — 파라미터 위치**

In [ ]:
for name, p in model.named_parameters():              # ④ 파라미터 이름과 텐서를 함께 받음
    print(name, tuple(p.shape), p.numel())
print('합계', sum(p.numel() for p in model.parameters()))

### 7.2 손실 함수와 옵티마이저

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   # ① GPU가 있으면 GPU
model = model.to(device)                                                  # ② 모델을 그 장치로

criterion = nn.CrossEntropyLoss()                                         # ③ 분류용 손실
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)                 # ④ Day 2 7.4의 Adam
print(device)

**손실 실습 1단계 — 점수에서 손실까지**

In [ ]:
logits = torch.tensor([[2.1, -0.5, 0.3]])                          # ① 클래스별 점수 — 사진 1장
print(torch.softmax(logits, dim=1))                                # ② 확률
for label in [0, 1, 2]:
    print(label, criterion(logits, torch.tensor([label])).item())  # ③ 정답 번호별 손실

**손실 실습 2단계 — 학습 전의 손실**

In [ ]:
images, labels = next(iter(train_loader))                          # ④ 학습 배치 하나
images, labels = images.to(device), labels.to(device)
with torch.no_grad():
    outputs = model(images)
print(criterion(outputs, labels).item())                           # ⑤ 학습 전 손실
print(torch.softmax(outputs, dim=1).mean(0))                       # ⑥ 클래스별 평균 확률

### 7.3 학습 루프 — 검증 단계와 기록

In [ ]:
def train_one_epoch(loader):
    model.train()                                         # ① 학습 모드
    total_loss, correct, n = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)   # ② 배치를 모델과 같은 장치로
        outputs = model(images)                           # 1 순전파
        loss = criterion(outputs, labels)                 # 2 손실
        optimizer.zero_grad()                             # 3 기울기 초기화
        loss.backward()                                   # 4 역전파
        optimizer.step()                                  # 5 갱신
        total_loss += loss.item() * images.size(0)        # ③ 배치 손실 × 장 수 — 평균 복원용
        correct += (outputs.argmax(1) == labels).sum().item()   # ④ 맞힌 수
        n += images.size(0)
    return total_loss / n, correct / n

In [ ]:
def evaluate(loader):
    model.eval()                                          # ⑤ 평가 모드
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():                                 # ⑥ 기울기 계산 끔
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            n += images.size(0)
    return total_loss / n, correct / n

In [ ]:
trial = SmallCNN(num_classes=len(train_ds.classes)).to(device)   # ① 실험용 모델 — 본 학습의 model과 별개
trial_opt = torch.optim.Adam(trial.parameters(), lr=1e-3)
before = criterion(trial(images), labels)
trial_opt.zero_grad(); before.backward(); trial_opt.step()       # ② 한 번 갱신
with torch.no_grad():
    after = criterion(trial(images), labels)                     # ③ 같은 배치로 다시 계산
print(before.item(), after.item())

**학습 실습 2단계 — epoch 반복과 기록**

> 저장 위치가 Google Drive입니다. Day 1 5.2의 Drive 연결 셀을 먼저 실행합니다.

In [ ]:
EPOCHS = 15
SAVE = '/content/drive/MyDrive/day3_best.pth'             # ⑦ Drive — 세션이 끝나도 남음
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val = float('inf')

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_one_epoch(train_loader)
    va_loss, va_acc = evaluate(val_loader)
    for k, v in zip(history, [tr_loss, va_loss, tr_acc, va_acc]):
        history[k].append(v)                              # ⑧ 4개 리스트에 기록
    if va_loss < best_val:                                # ⑨ 검증 손실 최소 갱신 시
        best_val = va_loss
        torch.save(model.state_dict(), SAVE)              # ⑩ 그 시점의 가중치 저장
    print(f'[{epoch+1:2d}/{EPOCHS}] train {tr_loss:.3f} / {tr_acc:.1%}   val {va_loss:.3f} / {va_acc:.1%}')

### 7.4 배치·epoch·학습률의 기준값

> GPU가 배정되지 않은 경우에 사용하는 축소 설정입니다(강의 자료 7.4). 실행하면 학습 로더가 130장의 부분 데이터로 교체됩니다.

In [ ]:
from torch.utils.data import Subset
small_idx = list(range(0, len(train_ds), 8))          # ① 8장마다 1장 — 클래스 순서로 정렬되어 있어 세 클래스에서 고르게 선택
train_loader = DataLoader(Subset(train_ds, small_idx), batch_size=16, shuffle=True, num_workers=2)
print(len(small_idx), len(train_loader))              # → 130 9

### 7.5 재현성

In [ ]:
import random, numpy as np, torch
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)   # 세 라이브러리의 난수를 고정

### 7.6 응용 실습 — 손실이 1 근처에서 멈춘 모델

> 응용 실습 7.6에서 비교하는 `forward`의 마지막 줄입니다. 따로 실행하지 않습니다.

```python
        return torch.softmax(self.classifier(x), dim=1)   # 점수 대신 확률을 반환
```

## 8. 결과 해석

### 8.1 학습 곡선 — 4유형 진단

**참고 코드**

In [ ]:
import matplotlib.pyplot as plt
epochs = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(epochs, history['train_loss'], label='train')
ax1.plot(epochs, history['val_loss'],   label='val')
ax1.set_title('loss'); ax1.legend()
ax2.plot(epochs, history['train_acc'], label='train')
ax2.plot(epochs, history['val_acc'],   label='val')
ax2.set_title('accuracy'); ax2.legend()
plt.show()

### 8.2 조기 종료와 최적 epoch 저장

In [ ]:
best_epoch = history['val_loss'].index(min(history['val_loss'])) + 1
print('최적 epoch =', best_epoch, '/ 검증 손실 =', round(min(history["val_loss"]), 3))

### 8.4 혼동 행렬 읽기

In [ ]:
import numpy as np
classes = val_ds.classes
K = len(classes)
cm = np.zeros((K, K), dtype=int)                          # ① K×K 정수 행렬(전부 0)
wrong = []                                                # 8.6용 — 틀린 사례 보관

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        preds = model(images.to(device)).argmax(1).cpu()  # ② 예측 번호 — CPU로 되돌림
        for img, t, p in zip(images, labels, preds):
            cm[t, p] += 1                                 # ③ (실제, 예측) 칸에 1 추가
            if t != p:
                wrong.append((img, t.item(), p.item()))

print(' ' * 18 + ''.join(f'{c:>18s}' for c in classes))   # 열 머리 = 예측 클래스
for i, row in enumerate(cm):
    print(f'{classes[i]:>18s}' + ''.join(f'{v:18d}' for v in row))   # 행 = 실제 클래스

### 8.5 정밀도·재현율·F1 — 병해 판정에서 더 위험한 오류

In [ ]:
for k, c in enumerate(classes):
    tp = cm[k, k]
    precision = tp / cm[:, k].sum() if cm[:, k].sum() else 0   # ④ 열 합 — 0 나눗셈 방지
    recall    = tp / cm[k, :].sum()                             # ⑤ 행 합
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    print(f'{c:>18s}  정밀도 {precision:.1%}  재현율 {recall:.1%}  F1 {f1:.1%}')

### 8.6 오분류 사례 관찰

In [ ]:
def denorm(t):                                            # ⑥ Normalize를 되돌려 표시용으로
    mean = torch.tensor(MEAN).view(3, 1, 1); std = torch.tensor(STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0)  # (C,H,W) → (H,W,C)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (img, t, p) in zip(axes.flat, wrong[:8]):        # ⑦ 최대 8장
    ax.imshow(denorm(img)); ax.axis('off')
    ax.set_title(f'true {classes[t]}\npred {classes[p]}', fontsize=9)   # ⑧ 영문 — Colab 기본 글꼴에 한글 없음
plt.show()
print('오분류', len(wrong), '장 /', len(val_ds), '장')

### 8.9 응용 실습 — 시험 세트로 최종 확인

**② 최적 가중치 불러오기**

In [ ]:
best_model = SmallCNN(num_classes=len(classes)).to(device)            # 같은 구조의 새 모델
best_model.load_state_dict(torch.load(SAVE, map_location=device))     # 7.3이 저장한 최적 epoch 가중치
best_model.eval()

## 9. Day 산출물 — 콩잎 분류기와 직접 선택한 데이터

### 9.2 필수 ① — 콩잎 절차와 저장 확인

In [ ]:
import os
print(os.path.exists(SAVE), round(os.path.getsize(SAVE) / 1e6, 2), 'MB')   # ① 존재 여부·크기
state = torch.load(SAVE, map_location='cpu')
print(list(state.keys()))                                 # ② 저장된 층 이름 목록

### 9.3 필수 ② — 데이터 고르기

**데이터 실습 1단계 — 내려받기 전 구성 확인**

In [ ]:
from datasets import load_dataset_builder
NAME = "Project-AgML/apple_leaf_disease_classification"   # ① 고른 데이터셋 이름 — 예시 = 사과잎 병해 3종
b = load_dataset_builder(NAME)                             # ② 데이터는 받지 않고 설명 정보만 읽음
print(b.info.features)                                     # ③ 열 이름과 형식 · 클래스 목록
print({k: v.num_examples for k, v in b.info.splits.items()})   # ④ 분할별 장 수
print(b.info.download_size)                                # ⑤ 내려받을 용량(바이트)

### 9.4 필수 ② — 코드에서 바꿀 곳

**데이터 실습 2단계 — 불러오기·나누기·내보내기**

In [ ]:
import os
from datasets import load_dataset
ds = load_dataset(NAME)                                    # ① 1단계에서 고른 데이터셋
LABEL = 'label'                                            # ② 라벨 열 이름 — 1단계 ③에서 확인
names = ds['train'].features[LABEL].names

s1 = ds['train'].train_test_split(test_size=0.3, seed=0, stratify_by_column=LABEL)   # ③ 70% 학습 / 30% 나머지
s2 = s1['test'].train_test_split(test_size=0.5, seed=0, stratify_by_column=LABEL)    # ④ 나머지를 반씩
parts = {'train': s1['train'], 'val': s2['train'], 'test': s2['test']}               # ⑤ 폴더 이름 → 분할

ROOT = '/content/data2'                                    # ⑥ 콩잎과 다른 위치
for folder, part in parts.items():
    for i, row in enumerate(part):
        dst = os.path.join(ROOT, folder, names[row[LABEL]])
        os.makedirs(dst, exist_ok=True)
        row['image'].save(os.path.join(dst, f'{i:04d}.jpg'), quality=95)

### 9.6 도달 — 클래스 가중치

**가중치 실습 1단계 — 클래스별 장 수와 역수 가중치**

In [ ]:
K = len(classes)
counts = torch.tensor([sum(1 for _, y in train_ds.samples if y == k) for k in range(K)],
                      dtype=torch.float)                  # ① 클래스별 학습 장 수
inv = (1.0 / counts); inv = inv / inv.sum() * K           # ② 장 수의 역수 — 평균 1로 정규화
print(dict(zip(classes, counts.tolist())))
print(dict(zip(classes, [round(w, 2) for w in inv.tolist()])))

**가중치 실습 2단계 — 대상 클래스 지정과 재학습**

In [ ]:
k_low = 0                                                  # ③ 8.5에서 재현율이 가장 낮은 클래스 번호 — 실행 예 = 0(angular_leaf_spot)
weight = torch.ones(K); weight[k_low] = 2.0                # ④ 그 클래스의 오류만 2배로 계산
criterion = nn.CrossEntropyLoss(weight=weight.to(device)) # ⑤ 가중 손실로 교체
model = SmallCNN(K).to(device)                             # ⑥ 새 모델로 다시 학습
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### 9.7 도전 — 증강 유무 비교

In [ ]:
train_ds_plain = datasets.ImageFolder(f'{ROOT}/train', transform=val_tf)   # ① 학습에도 결정론 변환
train_loader_plain = DataLoader(train_ds_plain, batch_size=32, shuffle=True, num_workers=2)

### 9.9 응용 실습 — 검증 폴더에 한 클래스만 있는 결과

> 응용 실습 9.9에서 원인을 찾는 대상 코드입니다.

In [ ]:
n = len(ds['train'])
cut = int(n * 0.7)
parts = {'train': ds['train'].select(range(0, cut)),     # 앞 70%
         'val':   ds['train'].select(range(cut, n))}     # 뒤 30%

## 11. 보충

### 11.1 전이 학습 — 소량 데이터의 표준 접근

In [ ]:
from torchvision import models
model_conv = models.resnet18(weights='IMAGENET1K_V1')     # ① ImageNet 사전학습 ResNet18
for param in model_conv.parameters():
    param.requires_grad = False                           # ② 전 층 동결 — 기울기를 계산하지 않음
num_ftrs = model_conv.fc.in_features
model_conv.fc = nn.Linear(num_ftrs, len(classes))        # ③ 마지막 층만 새로 — 이 층만 requires_grad=True
optimizer = torch.optim.Adam(model_conv.fc.parameters(), lr=1e-3)   # ④ 옵티마이저에는 새 층만

### 11.4 학습 없이 분류하기 — CLIP 제로샷과 대비

> 처음 실행할 때 `!pip install open_clip_torch`로 라이브러리를 먼저 설치합니다. 사전학습 모델(약 600MB)을 내려받습니다.

In [ ]:
# pip install open_clip_torch  — 처음 실행할 때 먼저 설치
import open_clip
clip_model, _, clip_tf = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
tokenizer = open_clip.get_tokenizer('ViT-B-32')
prompts = ['a photo of a bean leaf with angular leaf spot', 'a photo of a bean leaf with rust',
           'a photo of a healthy bean leaf']              # ① classes 순서(angular_leaf_spot·bean_rust·healthy)와 같게
text = tokenizer(prompts)
with torch.no_grad():
    text_feat = clip_model.encode_text(text)
    text_feat /= text_feat.norm(dim=-1, keepdim=True)     # ② 단위 벡터

### 11.6 클래스 불균형 — 가중치 외의 대응

In [ ]:
from torch.utils.data import WeightedRandomSampler
sample_w = [1.0 / counts[y] for _, y in train_ds.samples]        # ① 각 샘플의 뽑힐 가중치 = 클래스 장 수의 역수
sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)
train_loader_bal = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=2)   # ② shuffle 대신 sampler

### 11.7 학습률 스케줄러와 옵티마이저 선택

> 학습 루프에서 스케줄러를 호출하는 위치만 나타낸 코드이며 따로 실행하지 않습니다(강의 자료 11.7).

```python
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)   # 5 epoch마다 절반
for epoch in range(EPOCHS):
    ...                                                   # 7.3 학습·검증
    scheduler.step()                                      # epoch 끝에 한 번
```

## 12. 다음 시간

### 12.1 다음 시간 준비(선택) — Raspberry Pi 5에 onnxruntime 설치

**터미널 명령** — 노트북이 아니라 터미널에서 실행합니다.

```bash
# Raspberry Pi 5에서 — 약 5분 · 네트워크 필요
python3 -m venv ~/onnx-env                       # ① 가상환경 생성
source ~/onnx-env/bin/activate                   # ② 활성화 — 프롬프트 앞에 (onnx-env) 표시
pip install onnxruntime numpy pillow             # ③ 설치
python -c "import onnxruntime as ort; print(ort.__version__)"   # ④ 확인
```